<a href="https://colab.research.google.com/github/Mansik-04/assignments/blob/main/ensemble%20method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Can we use Bagging for regression problems?

Yes ✅. Bagging works for both classification and regression. In regression, Bagging averages predictions from multiple base regressors.

2. Difference between multiple model training and single model training

Single model: Uses one algorithm trained once → more variance or bias possible.

Multiple models (Ensemble): Combines many weak learners to reduce variance (Bagging) or bias (Boosting).

3. Feature randomness in Random Forest

Random Forest introduces feature randomness by selecting a random subset of features at each split, ensuring trees are less correlated.

4. OOB (Out-of-Bag) Score

An internal validation score in Bagging/Random Forest. Each tree is trained on a bootstrap sample; ~37% of data left out (OOB) is used to estimate performance without needing a separate validation set.

5. Measuring feature importance in Random Forest

Two common ways:

Gini Importance (Mean Decrease in Impurity).

Permutation Importance (drop-column method, evaluate accuracy drop).

6. Working principle of Bagging Classifier

Draw bootstrap samples from dataset.

Train base classifiers on each sample.

Aggregate predictions by majority vote.

7. Evaluate Bagging Classifier’s performance

Using accuracy, precision, recall, F1-score, ROC-AUC, or OOB score.

8. Bagging Regressor

Same as classifier, but outputs are averaged instead of majority vote.

9. Main advantage of ensemble techniques

Reduce variance & bias.

More robust & accurate than single models.

10. Main challenge of ensemble methods

Computationally expensive.

Harder to interpret.

Risk of overfitting if not tuned properly.

11. Key idea behind ensemble techniques

“Wisdom of the crowd”: combining multiple weak learners yields stronger predictions.

12. Random Forest Classifier

An ensemble of decision trees trained on bootstrap samples with feature randomness; outputs class by majority vote.

13. Main types of ensemble techniques

Bagging

Boosting (AdaBoost, XGBoost, Gradient Boosting)

Stacking

14. Ensemble learning in ML

A technique that combines multiple models (weak learners) to improve overall performance.

15. When to avoid ensemble methods

When interpretability is important.

When dataset is very small.

When computational resources are limited.

16. How Bagging reduces overfitting

By averaging predictions from multiple models, Bagging reduces variance, making model more stable.

17. Why Random Forest is better than single Decision Tree

Less variance.

Better generalization.

Handles noise better.

18. Role of bootstrap sampling in Bagging

Bootstrap sampling introduces diversity in training subsets, making models less correlated and ensemble stronger.

19. Real-world applications of ensemble techniques

Fraud detection

Medical diagnosis

Credit scoring

Sentiment analysis

Recommendation systems

20. Difference between Bagging and Boosting

Bagging: Parallel, reduces variance, independent learners, majority vote.

Boosting: Sequential, reduces bias, focuses on misclassified samples, weighted vote.

In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import BaggingClassifier, BaggingRegressor, RandomForestClassifier, RandomForestRegressor, StackingClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support
import seaborn as sns
import matplotlib.pyplot as plt

RANDOM_STATE = 42

# Dataset
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=RANDOM_STATE)

# 1. Bagging Classifier with Decision Trees
bag_clf = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=RANDOM_STATE)
bag_clf.fit(X_train, y_train)
print("Bagging Classifier Accuracy:", accuracy_score(y_test, bag_clf.predict(X_test)))

# 2. Bagging Regressor
house = fetch_california_housing()
Xh, yh = house.data, house.target
Xh_train, Xh_test, yh_train, yh_test = train_test_split(Xh, yh, test_size=0.2, random_state=RANDOM_STATE)
bag_reg = BaggingRegressor(estimator=DecisionTreeRegressor(), n_estimators=50, random_state=RANDOM_STATE)
bag_reg.fit(Xh_train, yh_train)
print("Bagging Regressor MSE:", mean_squared_error(yh_test, bag_reg.predict(Xh_test)))

# 3. Random Forest Classifier + feature importance
rf_clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, oob_score=True)
rf_clf.fit(X_train, y_train)
print("Random Forest Accuracy:", accuracy_score(y_test, rf_clf.predict(X_test)))
print("Feature Importances:", rf_clf.feature_importances_)
print("OOB Score:", rf_clf.oob_score_)

# 4. Random Forest Regressor vs Decision Tree
rf_reg = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
tree_reg = DecisionTreeRegressor(random_state=RANDOM_STATE)
rf_reg.fit(Xh_train, yh_train)
tree_reg.fit(Xh_train, yh_train)
print("Random Forest MSE:", mean_squared_error(yh_test, rf_reg.predict(Xh_test)))
print("Decision Tree MSE:", mean_squared_error(yh_test, tree_reg.predict(Xh_test)))

# 5. Bagging with SVM
bag_svm = BaggingClassifier(estimator=SVC(), n_estimators=10, random_state=RANDOM_STATE)
bag_svm.fit(X_train, y_train)
print("Bagging SVM Accuracy:", accuracy_score(y_test, bag_svm.predict(X_test)))

# 6. Random Forest with different n_estimators
for trees in [10, 50, 100, 200]:
    rf = RandomForestClassifier(n_estimators=trees, random_state=RANDOM_STATE)
    rf.fit(X_train, y_train)
    print(f"RF with {trees} trees Accuracy:", accuracy_score(y_test, rf.predict(X_test)))

# 7. Bagging with Logistic Regression (AUC)
bag_log = BaggingClassifier(estimator=LogisticRegression(max_iter=5000), n_estimators=10, random_state=RANDOM_STATE)
bag_log.fit(X_train, y_train)
probs = bag_log.predict_proba(X_test)[:,1]
print("Bagging Logistic Regression AUC:", roc_auc_score(y_test, probs))

# 8. Ensemble comparison: Bagging vs Random Forest
print("Bagging Acc:", accuracy_score(y_test, bag_clf.predict(X_test)))
print("Random Forest Acc:", accuracy_score(y_test, rf_clf.predict(X_test)))

# --- Advanced Practical ---

# 9. GridSearchCV with Random Forest
param_grid = {'n_estimators':[50,100], 'max_depth':[None,5,10]}
grid = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), param_grid, cv=3)
grid.fit(X_train, y_train)
print("Best RF Params:", grid.best_params_)

# 10. Bagging Regressor with different n_estimators
for n in [10,50,100]:
    bag_reg = BaggingRegressor(DecisionTreeRegressor(), n_estimators=n, random_state=RANDOM_STATE)
    bag_reg.fit(Xh_train, yh_train)
    print(f"BaggingRegressor {n} estimators MSE:", mean_squared_error(yh_test, bag_reg.predict(Xh_test)))

# 11. Stacking Classifier
stack_clf = StackingClassifier(
    estimators=[('dt', DecisionTreeClassifier()), ('svm', SVC(probability=True)), ('lr', LogisticRegression(max_iter=5000))],
    final_estimator=LogisticRegression(max_iter=5000)
)
stack_clf.fit(X_train, y_train)
print("Stacking Accuracy:", accuracy_score(y_test, stack_clf.predict(X_test)))

Bagging Classifier Accuracy: 0.956140350877193
Bagging Regressor MSE: 0.2572988359842641
Random Forest Accuracy: 0.9649122807017544
Feature Importances: [0.04870337 0.01359088 0.05326975 0.04755501 0.00728533 0.01394433
 0.06800084 0.10620999 0.00377029 0.00388577 0.02013892 0.00472399
 0.01130301 0.02240696 0.00427091 0.00525322 0.00938583 0.00351326
 0.00401842 0.00532146 0.07798688 0.02174901 0.06711483 0.15389236
 0.01064421 0.02026604 0.0318016  0.14466327 0.01012018 0.00521012]
OOB Score: 0.9560439560439561
Random Forest MSE: 0.2553684927247781
Decision Tree MSE: 0.495235205629094
Bagging SVM Accuracy: 0.9473684210526315
RF with 10 trees Accuracy: 0.956140350877193
RF with 50 trees Accuracy: 0.9649122807017544
RF with 100 trees Accuracy: 0.9649122807017544
RF with 200 trees Accuracy: 0.9649122807017544


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Bagging Logistic Regression AUC: 0.9980347199475925
Bagging Acc: 0.956140350877193
Random Forest Acc: 0.9649122807017544
Best RF Params: {'max_depth': None, 'n_estimators': 50}
BaggingRegressor 10 estimators MSE: 0.2824242776841025
BaggingRegressor 50 estimators MSE: 0.2572988359842641
BaggingRegressor 100 estimators MSE: 0.25592438609899626


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Stacking Accuracy: 0.9649122807017544


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
